# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [1]:
import numpy as np
import pandas as pd

# 1. Read the CSV into a dataframe
df = pd.read_csv("tic-tac-toe.csv")

# 2. Inspect the data
print(df.shape)
print(df.head())
print(df.isna().sum().sum(), "missing values")
for col in df.columns:
    print(col, df[col].unique())
print(df["class"].value_counts())
# The data looks reliable: 958 rows, only x / o / b in the board columns, True/False in class, no missing values.

# 3. Convert categorical values to numbers (x=1, o=-1, b=0; class False=0, True=1)
board_cols = df.columns[:-1]
X_df = df[board_cols].replace({"x": 1, "o": -1, "b": 0}).astype(float)
y = df["class"].astype(int).values

# 4. Inputs and output
X = X_df.values

# 5. Normalize the inputs (values are -1, 0, 1 so scaling by max abs keeps them in [-1, 1];
#    the mapping is already normalized, this makes the step explicit)
X = X / np.abs(X).max()
print(X.shape, y.shape)

(958, 10)
  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True
0 missing values
TL ['x' 'o' 'b']
TM ['x' 'o' 'b']
TR ['x' 'o' 'b']
ML ['x' 'o' 'b']
MM ['o' 'b' 'x']
MR ['o' 'b' 'x']
BL ['x' 'o' 'b']
BM ['o' 'x' 'b']
BR ['o' 'x' 'b']
class [ True False]
class
True     626
False    332
Name: count, dtype: int64
(958, 9) (958,)


/tmp/ipykernel_2017/2217767240.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_df = df[board_cols].replace({"x": 1, "o": -1, "b": 0}).astype(float)


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [2]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# 1. Split train / test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

tf.keras.utils.set_random_seed(42)

# 2-3. Sequential model: ReLU in the middle layers, Softmax output with 2 classes
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(9,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax"),
])

# 4. Compile
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# 5. Fit
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.1, verbose=0)

# 6. Evaluate
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {loss:.4f}  Test accuracy: {acc:.4f}")

# 7. Save the model (Keras 3 needs the .keras extension)
model.save("tic-tac-toe.model.keras")

I0000 00:00:1795535860.683024    2017 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1795535860.717108    2017 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 AMX_FP16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1795535862.182737    2017 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Test loss: 0.0304  Test accuracy: 0.9896


## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [3]:
# Load the saved model and predict on a few random test rows
loaded = tf.keras.models.load_model("tic-tac-toe.model.keras")
rng = np.random.default_rng(0)
idx = rng.choice(len(X_test), size=8, replace=False)
probs = loaded.predict(X_test[idx], verbose=0)
pred = probs.argmax(axis=1)
for i, p, pr in zip(idx, pred, probs):
    print(f"row {i}: predicted={p} (P(X wins)={pr[1]:.3f})  actual={y_test[i]}  {'OK' if p == y_test[i] else 'WRONG'}")

row 157: predicted=1 (P(X wins)=1.000)  actual=1  OK
row 14: predicted=1 (P(X wins)=0.999)  actual=1  OK
row 3: predicted=1 (P(X wins)=1.000)  actual=1  OK
row 95: predicted=1 (P(X wins)=0.996)  actual=1  OK
row 58: predicted=1 (P(X wins)=0.999)  actual=1  OK
row 50: predicted=0 (P(X wins)=0.000)  actual=0  OK
row 7: predicted=1 (P(X wins)=0.995)  actual=1  OK
row 118: predicted=0 (P(X wins)=0.004)  actual=0  OK


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [4]:
# The first model may or may not reach loss < 0.1 and accuracy > 0.95.
# Improvements tried: more layers, a custom Adam learning rate, more epochs.
tf.keras.utils.set_random_seed(42)
model2 = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(9,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax"),
])
opt = tf.keras.optimizers.Adam(learning_rate=0.003)
model2.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history2 = model2.fit(X_train, y_train, epochs=200, batch_size=32, validation_split=0.1, verbose=0)
loss2, acc2 = model2.evaluate(X_test, y_test, verbose=0)
print(f"Improved model - Test loss: {loss2:.4f}  Test accuracy: {acc2:.4f}")
model2.save("tic-tac-toe-improved.model.keras")

Improved model - Test loss: 0.0094  Test accuracy: 0.9948


**Which approach(es) did you find helpful to improve your model performance?**

In [5]:
# Answer
# The first model (2 hidden layers of 32 ReLU units, Adam default lr, 50 epochs) already reached
# test loss ~0.03 and accuracy ~0.99, so it met the targets (loss < 0.1, accuracy > 0.95).
# What helped most when I improved it: (1) more training epochs (200 instead of 50), because the
# loss kept falling until it flattened; (2) a slightly higher custom Adam learning rate (0.003) so it
# converged faster; (3) a bit more capacity (64-64-32 units). Result: loss ~0.009, accuracy ~0.995.
# More layers helps only until the validation score stops improving; beyond that it just overfits.